# starplast — download every input dataset

Reproduces the data behind the published cache. Everything is driven by
`starplast/datasets.py`, the single registry of provenance, so this notebook and the
methods section cannot drift apart.

**What this can and cannot do.** Public datasets with a stable URL are fetched
automatically: the ToxoDB identity tables, the four published CRISPR screens, the
proximity-labelling and pulldown corpora, and the PRIDE proteomics entries. Three
inputs cannot be fetched this way and are flagged at the end — GEO series that need
their processed supplementary matrix, the PubMed/PMC corpora (built by a separate
harvester), and the upstream node table.

Downloads are skipped when the target already exists, so re-running is cheap.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from starplast import datasets, paths

# Where things go is RESOLVED, not assumed. This notebook used to compute the dataset root as
# `Path.cwd().parent.parent`, which is true on one machine -- exactly the assumption paths.py was
# written to remove. Set $STARPLAST_DATA to put the tree somewhere else.
DATASETS = Path(paths.dataset_root(create=True))
print(paths.describe())
print(f"\ndownloads will go under {DATASETS}")

## 1. The registry itself

Every input, what it provides, and how much of the proteome it covers.


In [ ]:
import pandas as pd
t = datasets.as_table()
pd.set_option('display.max_colwidth', 42, 'display.width', 200)
t[['key','level','kind','provides','coverage','accession']]


Entries whose originating publication is not recorded. **Confirm these before citing.**


In [ ]:
for d in datasets.unresolved():
    print(f'{d.key:26s} {d.name}')


## 2. Fetch through the registry

`datasets.ensure(key)` is the fetcher, rather than a helper written out here. It knows the three
situations a dataset can be in -- a direct file, a GEO series whose FTP listing must be read rather than
its landing page, and a ToxoDB report that is a POST rather than a GET -- and it refuses what it cannot
honestly get instead of saving an HTML error page under a `.xlsx` name.

Every download is pinned with a SHA-256 on first fetch and checked against it afterwards, so a
publisher reissuing a supplement under the same URL is reported rather than absorbed.

In [ ]:
fetched, blocked = {}, {}
for d in datasets.REGISTRY:
    if not d.path:
        continue
    ok, how = datasets.fetchable(d.key)
    if not ok:
        blocked[d.key] = how
        continue
    p = datasets.ensure(d.key)
    fetched[d.key] = p

print(f"{sum(1 for p in fetched.values() if p)} fetched, "
      f"{sum(1 for p in fetched.values() if not p)} failed, {len(blocked)} cannot be fetched")

## 3. ToxoDB identity tables

Symbols, previous IDs and the GT1/VEG strain accessions. These are what lets a paper
citing a superseded accession still resolve to a current gene.


In [ ]:
import subprocess
print(subprocess.run([sys.executable,'-m','starplast.fetch_names'],
                     cwd=str(Path.cwd().parent), capture_output=True, text=True).stdout)


## 4. Published CRISPR screens

PMC's `/bin/` path returns 404 for these; the publishers serve them directly.


In [ ]:
SCREENS = {
  '37498952': [  # GRA17 synthetic-lethal (PLOS Pathogens)
    ('https://journals.plos.org/plospathogens/article/file?id=10.1371/journal.ppat.1011543.s001&type=supplementary',
     'gra17_synthlethal_PMC10409377_S1_phenotypes.xlsx'),
    ('https://journals.plos.org/plospathogens/article/file?id=10.1371/journal.ppat.1011543.s004&type=supplementary',
     'gra17_synthlethal_PMC10409377_S4_sgRNA_counts.xlsx')],
  '31481656': [  # in vivo CRISPR platform (Nat Commun)
    ('https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-019-11855-w/MediaObjects/41467_2019_11855_MOESM5_ESM.xlsx',
     'invivo_platform_PMC6722137_D2_phenotype_scores.xlsx'),
    ('https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-019-11855-w/MediaObjects/41467_2019_11855_MOESM6_ESM.xlsx',
     'invivo_platform_PMC6722137_D3_phenotype_scores.xlsx')],
  '40240328': [  # GRA12 strains / mouse subspecies (Nat Commun)
    ('https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-025-58876-2/MediaObjects/41467_2025_58876_MOESM5_ESM.xlsx',
     'gra12_PMC12003902_D3_gene_L2FC_screen1.xlsx'),
    ('https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-025-58876-2/MediaObjects/41467_2025_58876_MOESM6_ESM.xlsx',
     'gra12_PMC12003902_D4_gene_L2FC_screen2.xlsx')],
}

for pmid, files in SCREENS.items():
    print(f'PMID {pmid}')
    for url, name in files:
        fetch(url, DATASETS/'DNA'/'CRISPR_screen'/pmid/name)


The host-transcription screen ships its tables inside a nested archive, so it is
unpacked rather than saved directly.


In [ ]:
def fetch_epmc(pmcid, out_dir, keep=('.xlsx','.xls','.csv','.tsv')):
    """Europe PMC serves every supplementary file for an article as one archive."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    url = f'https://www.ebi.ac.uk/europepmc/webservices/rest/{pmcid}/supplementaryFiles'
    try:
        with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=300) as r:
            blob = r.read()
    except urllib.error.HTTPError as e:
        print(f'  {pmcid}: no supplementary files ({e.code})'); return 0
    n = 0
    def unpack(data, depth=0):
        nonlocal n
        try: z = zipfile.ZipFile(io.BytesIO(data))
        except zipfile.BadZipFile: return
        for nm in z.namelist():
            if nm.lower().endswith(keep):
                (out_dir/os.path.basename(nm)).write_bytes(z.read(nm)); n += 1
            elif nm.lower().endswith('.zip') and depth < 2:
                unpack(z.read(nm), depth+1)
    unpack(blob)
    print(f'  {pmcid}: {n} tables')
    return n

fetch_epmc('PMC12033024', DATASETS/'DNA'/'CRISPR_screen'/'37827122')


## 5. Proximity-labelling and pulldown corpora

97 studies with a tagged *Toxoplasma* protein, identified by screening all 33,924
abstracts for proximity-labelling (BioID / TurboID / APEX) or affinity-purification
vocabulary together with mass spectrometry. 81 are open access.

> These are **indexed but not yet parsed into edges**; do not describe them as
> integrated. Roughly 400 MB, so they are deliberately not committed to the repository.


In [ ]:
CATALOGUE = Path('catalogue_bioid_ipms.tsv')   # shipped beside this notebook
if not CATALOGUE.exists():
    print('catalogue not found -- see notebooks/README.md for how it was built')
else:
    import csv
    rows = [r for r in csv.DictReader(CATALOGUE.open(), delimiter='\t') if r['pmcid']]
    print(f'{len(rows)} open-access studies')
    for i, r in enumerate(rows, 1):
        sub = 'BioID' if r['kind'] == 'proximity' else 'IPMS'
        out = DATASETS/'post_translation'/sub/r['pmid']
        if (out/'META.json').exists():
            continue
        n = fetch_epmc(r['pmcid'], out)
        (out/'META.json').write_text(json.dumps(
            {k: r[k] for k in ('pmid','pmcid','year','kind','title')} | {'n_files': n}, indent=1))
        time.sleep(0.4)


## 6. Proteomics

PRIDE hosts the raw submissions; the per-protein iBAQ tables used here are the
publications' supplementary files. Listed for provenance, fetched where a stable URL exists.


In [ ]:
for d in datasets.registry(level='translation'):
    print(f'{d.key:18s} {d.accession}')
    print(f'   {d.url}')
    print(f'   note: {d.note}')


## 7. Inputs this notebook cannot fetch

Not a to-do list. These genuinely do not exist as downloadable files: some are only inside a paper's
supplementary section, one is deposited only as raw instrument output, and the literature corpora are
built rather than retrieved. `ensure()` names each one and returns `None` rather than producing a path
to something that is not the dataset.

In [ ]:
for key, why in sorted(blocked.items()):
    d = datasets.get(key)
    print(f"{key:24s} {why}")
    if d.citation:
        print(f"{'':24s}   {d.citation}")

## 8. Verify

Rebuild the cache and confirm the figures quoted in the methods section.


In [ ]:
ok, msg = paths.check()
print(msg)
missing = datasets.missing()
print(f"\n{len(datasets.REGISTRY)} registered datasets, {len(missing)} not present locally")
for k in missing:
    print("  ", k, datasets.fetchable(k)[1])

# Rebuild the cache from what is now on disk (~5 min):
#     python -m starplast.build_graph